# Gradient Boosting — Examples

This notebook introduces gradient boosting for classification and regression, then explores hyperparameters and feature importance.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

> Run cells from top to bottom. Test-set results are for final evaluation, not repeated hyperparameter selection.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

RANDOM_STATE = 42


## 1. Classification dataset
We'll use scikit-learn's built-in breast cancer dataset, so no download is required.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Features:", X.shape)
print("Class counts:")
print(y.value_counts().rename(index=dict(enumerate(dataset.target_names))))
display(X.head())


## 2. Train/test split and baseline gradient boosting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=RANDOM_STATE
)
gb_clf.fit(X_train, y_train)

train_pred = gb_clf.predict(X_train)
test_pred = gb_clf.predict(X_test)

results = pd.DataFrame({
    "Metric": ["Train accuracy", "Test accuracy", "Test precision (class 1)",
               "Test recall (class 1)", "Test F1 (class 1)"],
    "Score": [
        accuracy_score(y_train, train_pred),
        accuracy_score(y_test, test_pred),
        precision_score(y_test, test_pred),
        recall_score(y_test, test_pred),
        f1_score(y_test, test_pred),
    ]
})
display(results.round(4))
print(classification_report(y_test, test_pred, target_names=dataset.target_names))


## 3. Confusion matrix
In this dataset, class `0` is malignant and class `1` is benign. Read the class labels carefully when deciding which error is most costly.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, display_labels=dataset.target_names, cmap="Blues"
)
plt.title("Gradient Boosting — Confusion Matrix")
plt.tight_layout()
plt.show()


## 4. Compare a few hyperparameter configurations
This comparison is illustrative. In a real workflow, choose settings using cross-validation on training data and reserve the test set for final evaluation.

In [ ]:
configs = [
    {"name": "50 trees, lr=0.1", "n_estimators": 50, "learning_rate": 0.1},
    {"name": "150 trees, lr=0.1", "n_estimators": 150, "learning_rate": 0.1},
    {"name": "150 trees, lr=0.03", "n_estimators": 150, "learning_rate": 0.03},
]

comparison = []
for cfg in configs:
    model = GradientBoostingClassifier(
        n_estimators=cfg["n_estimators"],
        learning_rate=cfg["learning_rate"],
        max_depth=2,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    comparison.append({
        "Configuration": cfg["name"],
        "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
        "Test accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Test F1": f1_score(y_test, model.predict(X_test)),
    })

display(pd.DataFrame(comparison).round(4))


## 5. Gradient boosting vs. random forest
Both are tree ensembles, but they build trees differently. Compare them on the same split.

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=200, random_state=RANDOM_STATE, class_weight=None
)
rf_clf.fit(X_train, y_train)

model_comparison = pd.DataFrame([
    {
        "Model": "Gradient Boosting",
        "Accuracy": accuracy_score(y_test, gb_clf.predict(X_test)),
        "F1": f1_score(y_test, gb_clf.predict(X_test)),
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_clf.predict(X_test)),
        "F1": f1_score(y_test, rf_clf.predict(X_test)),
    },
])
display(model_comparison.round(4))


## 6. Feature importance
Impurity-based importance is useful for a first look, but it is not causal evidence and can be biased. Consider permutation importance on held-out data for a complementary view.

In [ ]:
importance = pd.Series(
    gb_clf.feature_importances_, index=X.columns
).sort_values(ascending=False)

display(importance.head(10).to_frame("Importance"))

importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Feature importance")
plt.title("Top 10 Gradient Boosting Features")
plt.tight_layout()
plt.show()


## 7. Regression example
We'll use a synthetic dataset to keep this notebook runnable offline.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=1500, n_features=10, noise=15, random_state=RANDOM_STATE
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

gb_reg = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.05, max_depth=2,
    random_state=RANDOM_STATE
)
gb_reg.fit(Xr_train, yr_train)
reg_pred = gb_reg.predict(Xr_test)

mae = mean_absolute_error(yr_test, reg_pred)
rmse = np.sqrt(mean_squared_error(yr_test, reg_pred))
r2 = r2_score(yr_test, reg_pred)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")


## 8. Cross-validation for model selection
Use training data only for this stage. The test set should be used once after the configuration has been selected.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=2,
    random_state=RANDOM_STATE
)
cv_scores = cross_val_score(candidate, X_train, y_train, cv=cv, scoring="f1")
print("CV F1 scores:", np.round(cv_scores, 4))
print(f"Mean CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


## Key takeaways

- Gradient boosting adds weak learners sequentially to reduce a loss function.
- Learning rate and number of estimators should be tuned together.
- Tree depth and leaf constraints control learner complexity.
- Use appropriate metrics, especially when classes are imbalanced or errors have different costs.
- Use cross-validation for model selection and keep the test set for final evaluation.
- Feature importance is a diagnostic, not a causal explanation.
